In [ ]:
# PLOTTING OF NEURAL NETWORK PREDICTIONS

import sys
import os

import numpy as np
import xarray as xr
import torch

import datetime as dt

import matplotlib as mpl
import matplotlib.pyplot as plt
import psyplot.project as psy
import cartopy.crs as ccrs
import colormaps as cmaps

from matplotlib.colors import ListedColormap
from matplotlib import cm
from matplotlib.ticker import FuncFormatter

utils_path = os.path.abspath('_utils')
if utils_path not in sys.path:
    sys.path.append(utils_path)

In [ ]:
# Select time steps

################################################################################################################

grid = 'n32'
igtype1 = 'IG'
igtype2 = 'SGIG16'

typename1 = ''
if igtype1=='IG':
    typename1 = ' '

t_start1 = dt.datetime(2022,7,1,0)
t_end1 = dt.datetime(2022,7,1,23)
t_delta1 = dt.timedelta(hours=1)

################################################################################################################

ts1 = []  # ts in datetime format

ts1.append(t_start1)

while ts1[-1] < t_end1:
    ts1.append(ts1[-1] + t_delta1)
    
    
tc1 = []        # tc is simple string
suffixes1 = []  # suffixes is T-Z-string
                                 
for i in range(len(ts1)):
    tc1.append(ts1[i].strftime('%Y%m%d%H%M%S'))
    suffixes1.append(ts1[i].strftime('%Y-%m-%d')+'T'+ts1[i].strftime('%H:%M:%S'))


ts1[0], tc1[0], suffixes1[0]

In [ ]:
# Select levels

levels = {1: '2 hPa',
          5: '10 hPa',
          8: '50 hPa',
          16: '250 hPa',
          21: '500 hPa'}

In [ ]:
# LOAD DATA PT 1

path1  = '/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_' + grid + '/' + igtype1 + '/'
files1 = []

for i in range(len(ts1)):
    month1=dt.datetime.strftime(ts1[i],'%Y-%m')
    files1.append(path1 + '__' + month1 + '/box_GLOBAL_70/' + ts1[i].strftime('%Y%m%d%H%M%S') + '.npz')

data1 = []
for i in range(len(ts1)):
    data1.append(np.load(files1[i]))

# Load NN and statistics

nn_path1 = '/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_n32/IG/_preprocessed_full2024_land_/'

stats1 = np.load(nn_path1 + '_statistics.npz')

len(data1), list(data1[0].keys()), list(stats1.keys()), len(stats1['feat_mean']), len(stats1['targ_mean'])

In [ ]:
# LOAD DATA PT 2

path2  = '/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_' + grid + '/' + igtype2 + '/'
files2 = []

for i in range(len(ts1)):
    month2=dt.datetime.strftime(ts1[i],'%Y-%m')
    files2.append(path2 + '__' + month2 + '/box_GLOBAL_70/' + ts1[i].strftime('%Y%m%d%H%M%S') + '_do.npz')

data2 = []
for i in range(len(ts1)):
    data2.append(np.load(files2[i]))

# Load NN and statistics

nn_path2 = '/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_n32/SGIG16/_preprocessed_full2024_land_/'

stats2 = np.load(nn_path2 + '_statistics.npz')

len(data2), list(data2[0].keys()), list(stats2.keys()), len(stats2['feat_mean']), len(stats2['targ_mean'])

In [ ]:
# DATA PREPARATION

nn_vec1 = []

for z in range(len(ts1)):
    nn_vec1.append(np.concatenate([data1[z]['ua'], data1[z]['va'], data1[z]['wa'], data1[z]['ta'],
                         np.expand_dims(data1[z]['geop'], axis=2), np.expand_dims(data1[z]['oro_std'], axis=2),
                         np.expand_dims(data1[z]['oro_anis'], axis=2), np.expand_dims(data1[z]['oro_angle'], axis=2),
                         np.expand_dims(data1[z]['oro_slope'], axis=2)], axis=2))
    
nn_vec2 = []

for z in range(len(ts1)):
    nn_vec2.append(np.concatenate([data2[z]['ua'], data2[z]['va'], data2[z]['wa'], data2[z]['ta'],
                         np.expand_dims(data2[z]['geop'], axis=2), np.expand_dims(data2[z]['oro_std'], axis=2),
                         np.expand_dims(data2[z]['oro_anis'], axis=2), np.expand_dims(data2[z]['oro_angle'], axis=2),
                         np.expand_dims(data2[z]['oro_slope'], axis=2)], axis=2))

len(nn_vec1), nn_vec1[0].shape, len(nn_vec2), nn_vec2[0].shape

In [ ]:
# NORMALIZATION

def normalization_feat(stats1, nn_vec):
    temp = nn_vec.reshape(-1,153)
    mean1_matrix = np.stack([stats1['feat_mean']]*temp.shape[0], axis=0)
    std1_matrix = np.stack([stats1['feat_std']]*temp.shape[0], axis=0)
    normalized = ((temp-mean1_matrix)/std1_matrix).reshape(50,128,153)
    return normalized

def denormalization_feat(stats1, nn_vec):
    temp = nn_vec.reshape(-1,153)
    mean1_matrix = np.stack([stats1['feat_mean']]*temp.shape[0], axis=0)
    std1_matrix = np.stack([stats1['feat_std']]*temp.shape[0], axis=0)
    denormalized = (temp*std1_matrix + mean1_matrix).reshape(50,128,153)
    return denormalized

def denormalization_targ(stats1, nn_vec):
    temp = nn_vec.reshape(-1,74)
    mean1_matrix = np.stack([stats1['targ_mean']]*temp.shape[0], axis=0)
    std1_matrix = np.stack([stats1['targ_std']]*temp.shape[0], axis=0)
    denormalized = (temp*std1_matrix + mean1_matrix).reshape(50,128,74)
    return denormalized

In [ ]:
# RESTRICT TO LAND / OROGRAPHY

orostd_threshold = np.partition(data1[0]['oro_std'].flatten(), 5003)[5003]
mask_land_sea = np.array([data1[0]['land_sea_mask'] == 0])[0,:,:]
mask_oro_std = np.array([data1[0]['oro_std'] < orostd_threshold])[0,:,:]
combined_mask = mask_oro_std

In [ ]:
# NN INFERENCE

def inference_NN(stats1, nn_vec1, model1, constraint=''):

    normalized_np = normalization_feat(stats1, nn_vec1).reshape(-1,153)
    normalized_torch = torch.tensor(normalized_np)
    prediction_torch = model1(normalized_torch)
    prediction_numpy = prediction_torch.detach().numpy()
    denormalized_prediction = denormalization_targ(stats1,prediction_numpy)

    if constraint is 'combined':
        denormalized_prediction[combined_mask] = 0
    elif constraint is 'land':
        denormalized_prediction[mask_land_sea] = 0

    return denormalized_prediction

In [ ]:
# LOADING OF NNs

model1 = torch.load(nn_path1 + '__models/UNet_2_2026-04-15_UNet_2_IG_land.pt', map_location=torch.device('cpu'), weights_only=False)
model2 = torch.load(nn_path2 + '__models/UNet_2_2026-04-15_UNet_2_SG_land.pt', map_location=torch.device('cpu'), weights_only=False)


truths1, predictions1, diffs1 = [], [], []
truths2, predictions2, diffs2 = [], [], []

constraint = 'land'    # land, combined

for z in range(len(ts1)):
    truths1.append(np.concatenate([data1[z]['zf'], data1[z]['mf']], axis = 2))
    if constraint is 'combined':
        truths1[z][combined_mask,:] = 0
    elif constraint is 'land':
        truths1[z][mask_land_sea,:] = 0

    predictions1.append(inference_NN(stats1, nn_vec1[z], model1, constraint=constraint))
    diffs1.append(predictions1[z]-truths1[z])
    
    truths2.append(np.concatenate([data2[z]['zf'], data2[z]['mf']], axis = 2))
    if constraint is 'combined':
        truths2[z][combined_mask,:] = 0
    elif constraint is 'land':
        truths2[z][mask_land_sea,:] = 0

    predictions2.append(inference_NN(stats2, nn_vec2[z], model2, constraint=constraint))
    diffs2.append(predictions2[z]-truths2[z])


# len(predictions1), predictions1[0].shape, len(truths1), truths1[0].shape, len(diffs1), diffs1[0].shape, len(predictions2), predictions2[0].shape, len(truths2), truths2[0].shape, len(diffs2), diffs2[0].shape,

In [ ]:
# ASSEMBLE DATASETS FOR PLOTTING

lat = np.linspace(70,-70,50)
lon = np.linspace(-180,180,128)
lev = 5


truths1_xr_lev, predictions1_xr_lev, diffs1_xr_lev = [], [], []

for z in range(len(ts1)):

    truths1_xr_lev.append(xr.Dataset(data_vars=dict(MF=(['lat', 'lon'], truths1[z][:,:,lev]*1000)),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon))))

    predictions1_xr_lev.append(xr.Dataset(data_vars=dict(MF=(['lat', 'lon'], predictions1[z][:,:,lev]*1000)),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon))))
    
    diffs1_xr_lev.append(xr.Dataset(data_vars=dict(MF=(['lat', 'lon'], diffs1[z][:,:,lev]*1000)),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon))))
    
truths1_xr_levavg, predictions1_xr_levavg, diffs1_xr_levavg = xr.zeros_like(truths1_xr_lev[0]), xr.zeros_like(predictions1_xr_lev[0]), xr.zeros_like(diffs1_xr_lev[0])

for z in range(len(ts1)):
    truths1_xr_levavg += truths1_xr_lev[z]
    predictions1_xr_levavg += predictions1_xr_lev[z]
    diffs1_xr_levavg += diffs1_xr_lev[z]

truths1_xr_levavg /= len(ts1)
predictions1_xr_levavg /= len(ts1)
diffs1_xr_levavg /= len(ts1)


truths2_xr_lev, predictions2_xr_lev, diffs2_xr_lev = [], [], []

for z in range(len(ts1)):

    truths2_xr_lev.append(xr.Dataset(data_vars=dict(MF=(['lat', 'lon'], truths2[z][:,:,lev]*1000)),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon))))

    predictions2_xr_lev.append(xr.Dataset(data_vars=dict(MF=(['lat', 'lon'], predictions2[z][:,:,lev]*1000)),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon))))
    diffs2_xr_lev.append(xr.Dataset(data_vars=dict(MF=(['lat', 'lon'], diffs2[z][:,:,lev]*1000)),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon))))
    
truths2_xr_levavg, predictions2_xr_levavg, diffs2_xr_levavg = xr.zeros_like(truths2_xr_lev[0]), xr.zeros_like(predictions2_xr_lev[0]), xr.zeros_like(diffs2_xr_lev[0])

for z in range(len(ts1)):
    truths2_xr_levavg += truths2_xr_lev[z]
    predictions2_xr_levavg += predictions2_xr_lev[z]
    diffs2_xr_levavg += diffs2_xr_lev[z]
    
truths2_xr_levavg /= len(ts1)
predictions2_xr_levavg /= len(ts1)
diffs2_xr_levavg /= len(ts1)

In [ ]:
# PLOTTING FUNCTION (2x2 PLOT)

plt.rcParams.update({'font.size': 29})

lonlatbox = None
cmap = cmaps.vik
cbar = 'r'
axis = None
scaling = None
clabel = 'mPa'
projection = None
savefig=False
savefigname=''
titlesize=32
fs2=29


def fourplot_1(plot00, plot01, plot10, plot11, title1='', title2='', title3='', title4=''):
    
    limits = 5#limits_dict[i]*1000
    bounds = np.linspace(-limits,limits,1000)

    fig = plt.figure(figsize=(30,12), dpi=150)

    width, height = 0.48, 0.42
    pad=18.5

    ax00 = fig.add_axes([0.04,  0.54,  width, height], projection=ccrs.PlateCarree())
    ax01 = fig.add_axes([0.545, 0.54,  width, height], projection=ccrs.PlateCarree())
    ax10 = fig.add_axes([0.04,  0.025, width, height], projection=ccrs.PlateCarree())
    ax11 = fig.add_axes([0.545, 0.025, width, height], projection=ccrs.PlateCarree())

    psy.plot.mapplot(plot00, ax=ax00, cmap=cmap, title=title1, cbar=cbar, clabel=clabel, bounds=bounds,
                     titlesize=titlesize, grid_labelsize=fs2, cticks=np.linspace(-limits, limits, 11), cticksize=fs2, clabelsize=fs2, extend='both')
    ax00.set_title(title1, pad=pad, fontsize=titlesize)

    psy.plot.mapplot(plot01, ax=ax01, cmap=cmap, title=title2, cbar=cbar, clabel=clabel, bounds=bounds,
                     titlesize=titlesize, grid_labelsize=fs2, cticks=np.linspace(-limits, limits, 11), cticksize=fs2, clabelsize=fs2, extend='both')
    ax01.set_title(title2, pad=pad, fontsize=titlesize)

    psy.plot.mapplot(plot10, ax=ax10, cmap=cmap, title=title3, cbar=cbar, clabel=clabel, bounds=bounds,
                     titlesize=titlesize, grid_labelsize=fs2, cticks=np.linspace(-limits, limits, 11), cticksize=fs2, clabelsize=fs2, extend='both')
    ax10.set_title(title3, pad=pad, fontsize=titlesize)

    psy.plot.mapplot(plot11, ax=ax11, cmap=cmap, title=title4, cbar=cbar, clabel=clabel, bounds=bounds,
                     titlesize=titlesize, grid_labelsize=fs2, cticks=np.linspace(-limits, limits, 11), cticksize=fs2, clabelsize=fs2, extend='both')
    ax11.set_title(title4, pad=pad, fontsize=titlesize)

   # fig.savefig("/p/project1/icon-a-ml/haslauer1/____project1/_publication_4500/_20260601_inference_land_07.png", dpi=150)


In [ ]:
# PLOT

fourplot_1(truths1_xr_levavg, predictions1_xr_levavg, truths2_xr_levavg, predictions2_xr_levavg,
            title1='ERA5 2022-07-01 average $MF_x$ at 10 hPa',
            title2='U-Net 2022-07-01 average $MF_x$ at 10 hPa',
            title3='ERA5 2022-07-01 average subgrid $MF_x$ at 10 hPa',
            title4='U-Net 2022-07-01 average subgrid $MF_x$ at 10 hPa')

In [ ]:
# R2 VALUES (PROFILE AND MAP)

# LOAD R2 PROFILES CALCULATED EARLIER

r2_vec_IG_train = np.load('/p/project1/icon-a-ml/haslauer1/____project1/gw/__git/_data/numpy_20260415_levs_IG_train_orog.npy')
r2_vec_IG_test = np.load('/p/project1/icon-a-ml/haslauer1/____project1/gw/__git/_data/numpy_20260415_levs_IG_test_orog.npy')
r2_vec_SG_train = np.load('/p/project1/icon-a-ml/haslauer1/____project1/gw/__git/_data/numpy_20260415_levs_SG_train_orog.npy')
r2_vec_SG_test = np.load('/p/project1/icon-a-ml/haslauer1/____project1/gw/__git/_data/numpy_20260415_levs_SG_test_orog.npy')

r2_vec_train = r2_vec_SG_train
r2_vec_test = r2_vec_SG_test

r2_vals = np.load('/p/project1/icon-a-ml/haslauer1/____project1/gw/__git/_data/numpy_20260415_map_IG_test_orog.npy')
#r2_vals = np.load('/p/project1/icon-a-ml/haslauer1/____project1/gw/__git/_data/numpy_20260415_map_SG_test_orog.npy')

In [ ]:
len(r2_vals)

In [ ]:
# R2 VALUES FOR MAP


r2_plot = np.zeros((50,128)) - np.ones((50,128))

count = 0
for i in range(50):
    for j in range(128):
        if not mask_oro_std[i,j]:
            r2_plot[i,j] = r2_vals[count]
            count += 1
       # else: r2_plot[i,j] = 0

# for i in range(50):
#     for j in range(128):
#         if not mask_land_sea[i,j]:
#             r2_plot[i,j] = r2_vals[count]
#             count += 1
#         else: r2_plot[i,j] = 0

r2_plot.shape, r2_plot.min(), r2_plot.max()

In [ ]:
def count_between_minus1_and_0(arr):
    """Counts entries in a 2D numpy array that are > -1 and < 0."""
    return np.sum((arr > -1) & (arr < 0))

def count_greater_minus1(arr):
    """Counts entries in a 2D numpy array that are > -1 and < 0."""
    return np.sum((arr > -1))

count_between_minus1_and_0(r2_plot)/count_greater_minus1(r2_plot)

In [ ]:
# MASKING

mask_xr = xr.Dataset(data_vars=dict(mask=(['lat', 'lon'], combined_mask[:,:])),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon)))
mask_ls_xr = xr.Dataset(data_vars=dict(mask_ls=(['lat', 'lon'], mask_land_sea[:,:])),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon)))
mask_std_xr = xr.Dataset(data_vars=dict(mask_std=(['lat', 'lon'], mask_oro_std[:,:])),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon)))
# mask_xr

In [ ]:
r2_xr = xr.Dataset(data_vars=dict(R2=(['lat', 'lon'], r2_plot[:,:])),
                        coords=dict(lat=('lat', lat),
                                    lon=('lon', lon)))
# r2_xr, r2_vals.min()

In [ ]:
# DEFINE COLORMAP

color = mpl.cm.get_cmap(cmaps.naviaw)

sequential_cmap_name = 'Greens'     #cmaps.navia_r # Choose your sequential colormap
N_colors = 1000                     # Number of colors in the map
zero_value = 0.0001                 # A tiny value slightly above zero to separate 0 from the sequential range

VMAX = 1

sequential_cmap = cm.get_cmap(sequential_cmap_name, N_colors)
new_colors = sequential_cmap(np.linspace(0, 1, N_colors))

ocean = color(0.2) #np.array([0, 0, 0.7, 0.55])
all_colors = np.vstack([ocean, new_colors])

# Create the final ListedColormap
custom_cmap = ListedColormap(all_colors, name='BlackZeroSequential')
custom_cmap_cont = ListedColormap(new_colors, name='BlackZeroSequential_0')

In [ ]:
# DEFINE ERA5 LEVELS

LEVS=37

era5_levels = [1.0, 2.0, 3.0, 5.0, 7.0, 
        10.0, 20.0, 30.0, 50.0, 70.0, 
        100.0, 125.0, 150.0, 175.0, 200.0, 225.0, 250.0,
        300.0, 350.0, 400.0, 450.0, 500.0, 550.0, 600.0, 650.0,
        700.0, 750.0, 775.0, 800.0, 825.0, 850.0, 875.0, 900.0, 925.0, 950.0, 975.0, 1000.0]

In [ ]:
# PLOT WITH 2 SUBPLOTS

fig = plt.figure(figsize=(30,10), dpi=150)

plt.rcParams.update({'font.size': 29})
fs1=32
fs2=29


width, height = 0.1, 0.85
pad=18.5

ax00 = fig.add_axes([0.05,  0.105,  0.24, height])
ax01 = fig.add_axes([0.34, 0.205,  0.65, height-0.1], projection=ccrs.PlateCarree())


formatter = FuncFormatter(lambda y, _: f'{int(y)}')

ax00.plot(r2_vec_train, era5_levels, label='Training set', color = color(0.7), linewidth=2.5)
ax00.plot(r2_vec_test, era5_levels, label='Test set 2022a', color = color(0.2), linewidth=2.5)

ax00.invert_yaxis()
ax00.grid()
ax00.set_yscale('log')
ax00.set_xlabel('$R^2$',fontsize=fs2)
ax00.set_ylabel('Pressure level [hPa]',fontsize=fs2, labelpad=0)
ax00.set_xlim([0,1])
ax00.set_ylim([1000,1])
ax00.set_xticks([0.2,0.4,0.6,0.8,1.0])
ax00.yaxis.set_major_formatter(formatter)
#ax[0].set_title('Full GW spectrum\n', fontsize=fs1)
ax00.tick_params(axis='both', which='major', labelsize=fs2)
ax00.legend(fontsize=fs2)

psy.plot.mapplot(r2_xr, ax=ax01, cmap=custom_cmap, cbar=False, clabel=clabel, bounds=np.linspace(0,1,10),
                    titlesize=titlesize, grid_labelsize=fs2,  cticksize=fs2, clabelsize=fs2)

# Norm and mappable for colorbar
norm = mpl.colors.BoundaryNorm(boundaries=np.linspace(0.0,1,100), ncolors=999)
mappable = mpl.cm.ScalarMappable(norm=norm, cmap=custom_cmap_cont)
mappable.set_array([])

# ---- MANUAL CAX POSITION BELOW AX01 ----
cax01 = fig.add_axes([0.54, 0.1, 0.25, 0.03])  # Adjust bottom and height as needed

cbar = fig.colorbar(
    mappable,
    cax=cax01,
    orientation='horizontal',
    ticks=[0,0.2,0.4,0.6,0.8,1.0]
)
#cbar.set_clim(0.3, 1.0)
cbar.set_label('$R^2$ on test set 2022a', fontsize=fs2, labelpad=6)
cbar.ax.tick_params(labelsize=fs2)


fig.savefig("/p/project1/icon-a-ml/haslauer1/____project1/_publication_4500/_20260601_r2_levs_map_SG_orog.png", dpi=150)